In [ ]:
from datetime import datetime
import os
from pathlib import Path

import gymnasium as gym
import gym_pusht
import imageio.v2 as imageio
import numpy as np
import pygame
import torch
from IPython.display import Markdown, Video, display
from lerobot.datasets import LeRobotDataset


In [ ]:
mode = "obstacle"
data_path = Path(os.environ.get("PUSHT_OUTPUT_DIR", "outputs/pusht-gym")) / "pusht_obstacle"
run_path = data_path / datetime.now().strftime("run_%Y%m%d_%H%M%S")
video_path = run_path / "obses"
video_path.mkdir(parents=True, exist_ok=True)

# PushT advances by one action every 0.1 simulated seconds.
recording_fps = 10
steps = 200
episodes = 50
scale = 30
startup_delay_seconds = 2.0
startup_ramp_steps = 5

env = gym.make(
    "gym_pusht/PushT-v0",
    render_mode="rgb_array",
    obs_type="pixels_agent_pos",
    scale=scale,
    max_episode_steps=300,
    enable_obstacle=True,
)
base_env = env.unwrapped

# Store policy inputs and absolute target actions in a LeRobot dataset.
dataset_root = run_path / "lerobot_dataset"
dataset = LeRobotDataset.create(
    repo_id=f"local/{mode}_mouse_{run_path.name}",
    root=dataset_root,
    fps=recording_fps,
    use_videos=True,
    features={
        "observation.image": {
            "dtype": "video",
            "shape": (96, 96, 3),
            "names": ["height", "width", "channels"],
        },
        "observation.state": {
            "dtype": "float32",
            "shape": (2,),
            "names": ["agent_x", "agent_y"],
        },
        "action": {
            "dtype": "float32",
            "shape": (2,),
            "names": ["target_x", "target_y"],
        },
    },
)

# The simulation is rendered into a separate window so rgb_array observations
# remain available to the recorder and the policy.
pygame.init()
pygame.display.set_caption("PushT mouse demonstration recorder")
window = pygame.display.set_mode((512, 512))
loop_clock = pygame.time.Clock()

abs_actions = np.zeros((episodes, steps, 2), dtype=np.float32)
rel_actions = np.zeros((episodes, steps, 2), dtype=np.float32)
agent_positions = np.zeros((episodes, steps, 2), dtype=np.float32)
full_states = np.zeros((episodes, steps, 5), dtype=np.float32)
rewards = np.zeros((episodes, steps), dtype=np.float32)
coverages = np.zeros((episodes, steps), dtype=np.float32)
successes = np.zeros((episodes, steps), dtype=np.bool_)
lengths = np.zeros(episodes, dtype=np.int64)

task = f"Push the T block with scale {scale} into the green goal zone using the mouse-controlled pusher."

for episode in range(episodes):
    observation, info = env.reset()

    # Give the operator time to place the cursor after reset, then ramp from
    # the agent's reset position to the cursor target to avoid a sudden jump.
    delay_frames = int(startup_delay_seconds * recording_fps)
    for _ in range(delay_frames):
        window.blit(base_env._draw(), (0, 0))
        pygame.event.pump()
        pygame.display.flip()
        loop_clock.tick(recording_fps)
    initial_action = np.asarray(observation["agent_pos"], dtype=np.float32).copy()

    episode_video_path = video_path / f"episode_{episode:03d}.mp4"

    with imageio.get_writer(
        episode_video_path,
        fps=recording_fps,
        codec="libx264",
        is_batch=False,
    ) as video:
        for step in range(steps):
            # Show the full-resolution simulation while keeping the 96x96 RGB
            # observation returned by the environment for the training dataset.
            window.blit(base_env._draw(), (0, 0))
            pygame.event.pump()
            pygame.display.flip()

            mouse_x, mouse_y = pygame.mouse.get_pos()
            # Pymunk coordinates have their origin at the bottom-left; PyGame's
            # window coordinates have their origin at the top-left.
            mouse_action = np.array(
                [mouse_x, mouse_y],
                dtype=np.float32,
            )
            if step < startup_ramp_steps:
                ramp = (step + 1) / startup_ramp_steps
                abs_action = initial_action + ramp * (mouse_action - initial_action)
            else:
                abs_action = mouse_action

            abs_action = np.clip(abs_action, env.action_space.low, env.action_space.high)

            agent_pos = np.asarray(observation["agent_pos"], dtype=np.float32)
            full_state = np.array(
                [
                    *base_env.agent.position,
                    *base_env.block.position,
                    base_env.block.angle % (2 * np.pi),
                ],
                dtype=np.float32,
            )
            image = np.asarray(observation["pixels"], dtype=np.uint8)

            # Each LeRobot frame contains the observation before this action
            # and the exact absolute target sent to env.step().
            dataset.add_frame(
                {
                    "observation.image": image,
                    "observation.state": agent_pos,
                    "action": abs_action,
                    "task": task,
                }
            )

            # Capture the scene drawn by the environment.
            frame = base_env._get_img(
                base_env._draw(), width=224, height=224, render_action=False
            )
            video.append_data(frame)
            # video.append_data(image)

            abs_actions[episode, step] = abs_action
            rel_actions[episode, step] = abs_action - agent_pos
            agent_positions[episode, step] = agent_pos
            full_states[episode, step] = full_state

            observation, reward, terminated, truncated, info = env.step(abs_action)
            rewards[episode, step] = reward
            coverages[episode, step] = info["coverage"]
            successes[episode, step] = info["is_success"]
            lengths[episode] = step + 1

            # Keep mouse demonstrations at the environment's 10 Hz action rate.
            loop_clock.tick(recording_fps)
            if terminated or truncated:
                break

    dataset.save_episode()

    # Preserve the environment's complete state and outcome information in a
    # sidecar file; only the image, agent position, and target action are policy
    # features in the LeRobot dataset.
    episode_length = lengths[episode]
    np.savez_compressed(
        run_path / f"episode_{episode:03d}_metadata.npz",
        agent_positions=agent_positions[episode, :episode_length],
        full_states=full_states[episode, :episode_length],
        absolute_actions=abs_actions[episode, :episode_length],
        relative_actions=rel_actions[episode, :episode_length],
        rewards=rewards[episode, :episode_length],
        coverages=coverages[episode, :episode_length],
        successes=successes[episode, :episode_length],
        length=np.array(episode_length, dtype=np.int64),
        scale=np.array(scale, dtype=np.int64),
    )

dataset.finalize()
env.close()
pygame.quit()

# Also save compact tensors for compatibility with the existing analysis code.
torch.save(torch.from_numpy(abs_actions), run_path / "abs_actions.pth")
torch.save(torch.from_numpy(rel_actions), run_path / "rel_actions.pth")
torch.save(torch.from_numpy(full_states), run_path / "states.pth")
torch.save(torch.from_numpy(lengths), run_path / "seq_lengths.pth")

# print(f"LeRobot dataset: {dataset_root}")
# print(f"Recorded episodes: {lengths.tolist()}")


In [ ]:
for episode in range(episodes):
    display(Markdown(f"### Episode {episode:03d} — {mode}, T scale {scale}"))
    display(
        Video(
            filename=str(video_path / f"episode_{episode:03d}.mp4"),
            embed=True,
            width=400,
            html_attributes='controls loop muted playsinline style="max-width:100%;height:auto"',
        )
    )
